# Backtest fuera de la muestra

Hasta ahora he optimizado y evaluado con los mismos datos, lo que favorece a la cartera óptima. Aquí nuevamente calculo los pesos solo con el periodo de entrenamiento (2016-2022) y mido su rentabilidad y su Sharpe en el periodo de prueba (2023-2025), que el optimizador no ha visto.

Comparo la cartera de pesos iguales con la de máximo Sharpe con distintos valores de Z (credibilidad de las medias históricas).

In [20]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.portfolio import medias_shrinkage, cartera_max_sharpe, backtest

precios = pd.read_csv("../data/precios.csv", index_col="Date", parse_dates=True)
rentabilidades = precios.pct_change().dropna()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### División de los datos

Entrenamiento: 2016-2022 (1.761 días). Prueba: 2023-2025 (752 días). Los pesos se calculan solo con el primer tramo.

In [21]:
entrenamiento = rentabilidades.loc["2016":"2022"]
prueba = rentabilidades.loc["2023":"2025"]
print(entrenamiento.shape, prueba.shape)
print(entrenamiento.index[[0, -1]], prueba.index[[0, -1]])

(1761, 6) (752, 6)
DatetimeIndex(['2016-01-05', '2022-12-30'], dtype='datetime64[us]', name='Date', freq=None) DatetimeIndex(['2023-01-03', '2025-12-31'], dtype='datetime64[us]', name='Date', freq=None)


### Tasa libre de riesgo

Uso el rendimiento medio de las letras del Tesoro a 3 meses (serie DGS3MO, en %, dividida entre 100). La media de 2016-2022 (`rf_ent`) sirve para optimizar y la media de 2023-2025 (`rf_prueba`) para evaluar el Sharpe obtenido en la prueba. Calcular cada una solo con su tramo evita usar información del futuro al elegir los pesos.

In [22]:
tbill = pd.read_csv("../data/DGS3MO.csv", index_col=0, parse_dates=True).iloc[:, 0]
tbill = pd.to_numeric(tbill, errors="coerce")

rf_ent = tbill.loc["2016":"2022"].mean() / 100
rf_prueba = tbill.loc["2023":"2025"].mean() / 100
rf_ent, rf_prueba

(np.float64(0.011173885714285714), np.float64(0.048899332443257684))

### Pesos con datos de entrenamiento

Pesos de la cartera de máximo Sharpe estimados solo con 2016-2022, para distintos valores de Z, junto con la cartera de pesos iguales como referencia.

In [23]:
zs = [0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 1]

cov_ent = entrenamiento.cov().values * 252

pesos = {}
for z in zs:
    mu = medias_shrinkage(entrenamiento, z, rf_ent).values
    pesos[f"Z = {z}"] = cartera_max_sharpe(cov_ent, mu, rf_ent)
pesos["Pesos iguales"] = np.repeat(1 / entrenamiento.shape[1], entrenamiento.shape[1])

tabla_pesos = pd.DataFrame(pesos, index=entrenamiento.columns)
tabla_pesos.round(3)

,Z = 0,Z = 0.05,Z = 0.1,Z = 0.15,Z = 0.2,Z = 0.25,Z = 0.5,Z = 1,Pesos iguales
AGG,0.545,0.499,0.449,0.387,0.321,0.248,0.000,0.000,0.167
EEM,0.078,0.055,0.029,0.004,0.000,0.000,0.000,0.000,0.167
EFA,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.167
GLD,0.222,0.254,0.288,0.327,0.364,0.405,0.539,0.519,0.167
SPY,0.113,0.170,0.232,0.282,0.315,0.347,0.461,0.481,0.167
VNQ,0.041,0.022,0.001,0.000,0.000,0.000,0.000,0.000,0.167


### Resultado de los pesos de entrenamiento

Con Z = 0 la cartera se apoya en AGG (54,5 %), que aporta más diversificación. Al subir Z, AGG cede su peso poco a poco a GLD y SPY: con Z = 0,25 vale el 24,8 %, y desde Z = 0,5 desaparece, quedando GLD (54 %) y SPY (46 %). Con Z = 1 los pesos son 52 % y 48 %.

EEM y VNQ solo aparecen con Z muy bajo y con pesos pequeños, y EFA nunca entra. Con siete años de datos, dar a las medias históricas solo un 50 % de credibilidad ya basta para que el optimizador abandone la cartera diversificada.

### Resultados en el periodo de prueba

Aplico los pesos de entrenamiento a 2023-2025 y calculo la rentabilidad anual, la volatilidad anual y el Sharpe de cada cartera, con `rf_prueba` (4,9 %) como tasa libre de riesgo.


In [24]:
resultados = {}
for nombre, w in tabla_pesos.items():
    r_cartera = prueba.values @ w.values
    rent = r_cartera.mean() * 252
    vol = r_cartera.std() * np.sqrt(252)
    resultados[nombre] = {"Rentabilidad": rent, "Volatilidad": vol,
                          "Sharpe": (rent - rf_prueba) / vol}

pd.DataFrame(resultados).T.round(3)

,Rentabilidad,Volatilidad,Sharpe
Z = 0,0.133,0.071,1.180
Z = 0.05,0.147,0.074,1.339
Z = 0.1,0.163,0.077,1.484
Z = 0.15,0.178,0.082,1.585
Z = 0.2,0.193,0.087,1.651
Z = 0.25,0.209,0.094,1.705
Z = 0.5,0.261,0.117,1.814
Z = 1,0.260,0.117,1.808
Pesos iguales,0.163,0.105,1.088


Todas las carteras optimizadas superan a la de pesos iguales (Sharpe 1,09), incluso la de Z = 0 (1,18), que además tiene menos volatilidad (7,1 % frente a 10,5 %). El Sharpe crece con Z y llega a su máximo en Z = 0,5 (1,81), prácticamente igual que en Z = 1.

En este periodo, fiarse de las medias históricas no perjudicó: el shrinkage redujo la volatilidad, pero también la rentabilidad. Con solo tres años de prueba (error del Sharpe ≈ 0,6) y un periodo muy favorable para el oro y la bolsa, la conclusión no es definitiva.

### Robustez: dos particiones temporales

Repito el backtest con la función `backtest` de `src/portfolio.py` en dos particiones distintas: entrenar con 2016-2022 y evaluar en 2023-2025, o entrenar con 2016-2019 y evaluar en 2020-2025. Si una cartera gana en las dos, la conclusión es más fiable que con una sola. Comparo el Sharpe de pesos iguales, mínima varianza y máximo Sharpe con distintos valores de Z.

In [25]:
zs = [0, 0.25, 0.5, 1]
res_b = backtest(rentabilidades, tbill, "2022", "2023", "2025", zs=zs)
res_a = backtest(rentabilidades, tbill, "2019", "2020", "2025", zs=zs)

pd.DataFrame({"2016-22 / 2023-25": res_b, "2016-19 / 2020-25": res_a}).round(3)

,2016-22 / 2023-25,2016-19 / 2020-25
Pesos iguales,1.087,0.501
Mín. varianza,0.098,-0.080
Máx. Sharpe (Z = 0),1.180,0.505
Máx. Sharpe (Z = 0.25),1.704,0.530
Máx. Sharpe (Z = 0.5),1.812,0.507
Máx. Sharpe (Z = 1),1.807,0.468


In [26]:
for periodo, anios in {"2023-2025": 3, "2020-2025": 6}.items():
    print(f"Error aproximado del Sharpe en {periodo}: {1 / np.sqrt(anios):.2f}")

Error aproximado del Sharpe en 2023-2025: 0.58
Error aproximado del Sharpe en 2020-2025: 0.41


### Resultado de la robustez

Con la partición 2016-2022 / 2023-2025, las carteras de máximo Sharpe superan a la de pesos iguales (1,09) y el máximo se alcanza con Z = 0,5 (1,81). Con la partición 2016-2019 / 2020-2025 todas las carteras rinden mucho peor (Sharpe ≈ 0,5) y las diferencias se borran: pesos iguales (0,50), Z = 0 (0,51) y Z = 0,5 (0,51) quedan empatados, y el máximo Sharpe sin ajustar (Z = 1) es el peor de los optimizados (0,47) y pierde frente a pesos iguales.

La cartera de mínima varianza tiene un Sharpe casi nulo en la primera partición (0,10) y negativo en la segunda (−0,08). Probablemente se concentra en bonos, que rindieron poco con tipos de interés altos, pero habría que comprobar sus pesos.

### Conclusión del backtest

Z = 0,5 es el único valor que iguala o supera al máximo Sharpe sin ajustar en las dos particiones, así que el shrinkage no perjudicó y en la segunda partición corrigió parte de la pérdida. Sin embargo, las diferencias de la segunda partición están dentro del error de estimación (con seis años de prueba, el error del Sharpe es ≈ 0,4). No puedo afirmar que optimizar supere de forma fiable a repartir a partes iguales: en la segunda partición ninguna cartera optimizada se separa claramente de los pesos iguales, y la diferencia está dentro del error de estimación.